In [1]:
import sys

In [2]:
import json
# import spacy

In [3]:
sys.path.append("../src")

In [4]:
from load_data.funciones_carga_datos import load_filter_dataset_HuggingFace
from conexion_Neo4j.conexion_Neo4j import ConexionNeo4j


c:\Users\andre\anaconda3\envs\kag_env1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


In [5]:
database_Neo = "2wiki.prueba.rebel.3"
conn_Neo4j = ConexionNeo4j(database_Neo)

# INSPECCION ENTIDADES EXTRAIDAS

RECUPERACION DE TODAS LAS ENTIDADES

In [6]:
all_entis = conn_Neo4j.extraer_all_entidades_neo4j()

In [7]:
all_entis

['best horror movie of 1982',
 'a day will come',
 'prince henry, duke of cumberland',
 'sherlock holmes',
 'moscow domodedovo airport',
 'malik muhammad khan',
 'fire down below',
 'dana blankstein',
 'committed',
 'just once a great lady',
 'balochistan province',
 'master stroke',
 'maria teresa, grand duchess of luxembourg',
 'glamour boy',
 'calcasieu parish',
 'soldier',
 'maurice campbell',
 'language management',
 'erbessa integra',
 'al-hasan ibn ali',
 'earthlink',
 'mecklenburg- vorpommern',
 'fatimah',
 '2006 film',
 'center point road',
 'theodred ii',
 'santa clara unified school district',
 'anurag singh',
 'bloody birthday',
 "evelyn's love adventures",
 "pushpadana girls' college",
 'henry lawes luttrell, 2nd earl of carhampton',
 'jack ryan conway',
 'ali khalifa rahuma',
 'the sporting duchess',
 'thomas smith',
 'flirting with fate',
 'santa maría de santa cruz de la serós',
 'kaga province',
 'the ultimate warrior',
 'eureka',
 'david and bathsheba',
 'starstruck',

In [11]:
len(all_entis)

3437

## Entidades con menos de 3 o 2 letras

Con 3 letras no se pueden desechar

In [19]:
for ent in all_entis:
    if len(ent) < 4 :
        print(ent)

Me
SCR
Man
USA
Ali
Goa
NBA
R&B
Pop
Die
3-D
ABC
FCS
KMT
K
EP
RKO
438
SVT
M
The
Fox
NBC
R
30
Ch


In [21]:
for ent in all_entis:
    if len(ent) < 3 :
        print(ent)

Me
K
EP
M
R
30
Ch


## Entidades con alguna palabra inferior a 3 letras

In [17]:
for ent in all_entis:
    if any(len(word) < 3 for word in ent.split()):
        print(ent)

Warsaw Pact invasion of Czechoslovakia
Mirza Ghulam Ahmad of Qadian
Night by the Seashore
Ek Paheli
Roman Catholic Diocese of Springfield
Rhescuporis IV
S. N. Mathur
Brown Eyed GirlJackie Wilson Said (I'm in Heaven When You Smile)Domino
The Damned Do n't Cry
Legend of the Condor Heroes
Weekend in Paradise
Where Was I?
Three Men and a Baby
Ramesses II
Hassan II University
Secrets of the Underground
Give It to Me
Princess Augusta of Saxe-Gotha
A Piece of Eden
Hajjiabad -e Kark
Three Men and a Little Lady
Six Days of the Condor
Transilvania TV
Wedding Night in Paradise
Mehmed II
Noura Khalifa Al Suwaidi
The Rise of Skywalker
Ich für dich, du für mich
Three Men and a Cradle
Ruang Talok 69
novel of the same title
Mr. Moto ’s Last Warning
Carnival of Souls
novel of the same name
Tiberius Julius Rhescuporis IV
Time of the Gypsies
The Trail of the Lonesome Pine
La torre de los siete jorobados
U Mobile
Rhoemetalces I
Battle of the Bands
The Da Vinci Code
J. P. McGowan
Edward J. Yates
At the Mov

# Extraccion de Nacionalidades

BUSQUEDA DE ENTIDADES DE NACIONALIDADES (NORP/GPE) Y DE PERSONAS(PERSON)

In [39]:
import spacy

# Load the lightweight pre-trained model
nlp = spacy.load("en_core_web_sm")

text = "Albert Gordon is Canadian and MArco Tokinashi is Japanese, both working for an American company in Tokyo."

doc = nlp(text)

# Iterate over recognized entities
for ent in doc.ents:
    if ent.label_ == "NORP":
        print(f"Nationality: {ent.text}")
    elif ent.label_ == "GPE":
        print(f"Location/Country: {ent.text}")

Nationality: Canadian
Nationality: Japanese
Nationality: American
Location/Country: Tokyo


In [40]:
doc.ents

(Albert Gordon, Canadian, MArco Tokinashi, Japanese, American, Tokyo)

In [48]:
for ent in doc.ents:
    print(ent)
    print(ent.text)
    print(ent.label_)
    print(ent.start)
    print(ent.end)
    print("-"*10)

Albert Gordon
Albert Gordon
PERSON
0
2
----------
Canadian
Canadian
NORP
3
4
----------
MArco Tokinashi
MArco Tokinashi
PERSON
5
7
----------
Japanese
Japanese
NORP
8
9
----------
American
American
NORP
14
15
----------
Tokyo
Tokyo
GPE
17
18
----------


# Emparejar Persona con su nacionalidad

Buscar persona y ver si tiene nacionalidad den las N siguientes tokens

In [47]:
doc

Albert Gordon is Canadian and MArco Tokinashi is Japanese, both working for an American company in Tokyo.

In [49]:
doc.ents

(Albert Gordon, Canadian, MArco Tokinashi, Japanese, American, Tokyo)

In [ ]:




doc = nlp(text)

In [8]:
def extraer_nacionalidades(text, nlp, n_window):

    doc = nlp(text)
    # N_tokens = 3
    triplets_nacionalidad = []
    nationality = [ent.text for ent in doc.ents if ent.label_=="NORP"]
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            window = min(ent.end + n_window, len(doc))
            for token in doc[ent.end:window]:
                if token.text in nationality:
                    triplets_nacionalidad.append((ent.text, "country of birth", token.text))
                    break
    return triplets_nacionalidad


In [5]:
import spacy

In [6]:
nlp = spacy.load("en_core_web_sm")
text = "Albert Gordon is Canadian and MArco Tokinashi is Japanese, both working for an American company in Tokyo."
n_window = 3

In [9]:
triplets_nacionalidad = extraer_nacionalidades(text, nlp, n_window)
triplets_nacionalidad

[('Albert Gordon', 'country of birth', 'Canadian'),
 ('MArco Tokinashi', 'country of birth', 'Japanese')]

## PRUEBA 2 WIKI

In [8]:
dataset_2Wiki = load_filter_dataset_HuggingFace("xanhho/2wikimultihopqa", 20, "train")

In [11]:
dataset_2Wiki[11]

{'_id': '95c98f6808e511ebbda4ac1f6bf848b6',
 'type': 'comparison',
 'question': 'Are Alison Skipper and Diane Gilliam Fisher from the same country?',
 'context': '[["Learning Disability Quarterly", ["Learning Disability Quarterly is a quarterly peer-reviewed academic journal that covers the field of special education.", "The editors-in-chief are Diane P. Bryant and Brian Bryant (University of Texas at Austin).", "The journal was established in 1978 and is published by SAGE Publications on behalf of the Hammill Institute on Disabilities."]], ["Fisher and Schwartz cheating scandal", ["< onlyinclude>", "In August 2015, Boye Brogeland\'s team( Richard Schwartz, Allan Graves, Boye Brogeland, Espen Lindqvist, Huub Bertens, Daniel Korbel) lost in the quarter- finals of the Spingold to Jimmy Cayne\'s team( James Cayne, Michael Seamon, Lotan Fisher, Ron Schwartz, Alfredo Versace, Lorenzo Lauria) by 1 IMP following an appeal that lost his team 2 IMPs.", "The appeal involved Lotan Fisher and Ron 

### Busqueda entidades de lugares y de personas.

Salen muchisimas

In [30]:
for frase in json.loads(dataset_2Wiki[11]['context']):
    frase_txt = " ".join(frase[1])
    print(frase_txt)
    doc = nlp(frase_txt)

    # Iterate over recognized entities
    for ent in doc.ents:
        if ent.label_ == "NORP":
            print(f"Nationality: {ent.text}")
        elif ent.label_ == "GPE":
            print(f"Location/Country: {ent.text}")
        elif ent.label_ == "PERSON":
            print(f"Person: {ent.text}")
    print("-"*20)

Learning Disability Quarterly is a quarterly peer-reviewed academic journal that covers the field of special education. The editors-in-chief are Diane P. Bryant and Brian Bryant (University of Texas at Austin). The journal was established in 1978 and is published by SAGE Publications on behalf of the Hammill Institute on Disabilities.
Person: Diane P. Bryant
Person: Brian Bryant
Location/Country: Austin
--------------------
< onlyinclude> In August 2015, Boye Brogeland's team( Richard Schwartz, Allan Graves, Boye Brogeland, Espen Lindqvist, Huub Bertens, Daniel Korbel) lost in the quarter- finals of the Spingold to Jimmy Cayne's team( James Cayne, Michael Seamon, Lotan Fisher, Ron Schwartz, Alfredo Versace, Lorenzo Lauria) by 1 IMP following an appeal that lost his team 2 IMPs. The appeal involved Lotan Fisher and Ron Schwartz, Brogeland's teammates from the previous year when they won the Spingold. Brogeland spent the following day reviewing the Vugraph records from the quarter- final

### RELACION PERSONAS CON SU NACIONALIDAD.

Si entidad NORP cerca de entidad PERSON --> Citizenship

In [12]:
def extraer_nacionalidades(text, nlp, n_window):

    doc = nlp(text)
    # N_tokens = 3
    triplets_nacionalidad = []
    window_ctxt = []
    nationality = [ent.text for ent in doc.ents if ent.label_=="NORP"]
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            window = min(ent.end + n_window, len(doc))
            for token in doc[ent.end:window]:
                if token.text in nationality:
                    triplets_nacionalidad.append((ent.text, "originally from", token.text))
                    # triplets_nacionalidad.append((ent.text, "country of origin", token.lemma_))
                    window_ctxt.append(f"{ent.text} {doc[ent.end:window]}")
                    break
    return triplets_nacionalidad, window_ctxt

In [13]:
nlp = spacy.load("en_core_web_sm")
n_window = 4

In [14]:
all_triples_nacionalidad = []
for reg in dataset_2Wiki:
    # for frase in json.loads(dataset_2Wiki[11]['context']):
    for frase in json.loads(reg['context']):
        frase_txt = " ".join(frase[1])
        # print(frase_txt)
        triplets_nacionalidad, window_ctxt = extraer_nacionalidades(frase_txt, nlp, n_window)
        if triplets_nacionalidad:
            all_triples_nacionalidad += triplets_nacionalidad
            print(triplets_nacionalidad)
            print(window_ctxt)
            print("-"*20)

[('Ian Barry', 'originally from', 'Australian')]
['Ian Barry is an Australian director']
--------------------
[('Peter Levin', 'originally from', 'American')]
['Peter Levin is an American director']
--------------------
[('Sergio Mimica- Gezzan', 'originally from', 'American')]
['Sergio Mimica- Gezzan is an American film']
--------------------
[('Pamela Jain', 'originally from', 'Indian')]
['Pamela Jain is an Indian playback']
--------------------
[('Peter Levin', 'originally from', 'American')]
['Peter Levin is an American director']
--------------------
[('Ian Barry', 'originally from', 'Australian')]
['Ian Barry is an Australian director']
--------------------
[('Kannada', 'originally from', 'Malayalam')]
['Kannada cinema, and Malayalam']
--------------------
[('Tarcisio Fusco', 'originally from', 'Italian')]
['Tarcisio Fusco was an Italian composer']
--------------------
[('Walter Ulfig', 'originally from', 'German')]
['Walter Ulfig was a German composer']
--------------------
[('B

In [15]:
print(len(all_triples_nacionalidad))
print(len(list(set(all_triples_nacionalidad))))

27
15


In [16]:
for el in list(set(all_triples_nacionalidad)):
    print(el)

('Ola Cullin', 'originally from', 'Swedish')
('Bradford', 'originally from', 'American')
('Presley', 'originally from', 'acoustic')
('Peter Levin', 'originally from', 'American')
('Gary Hampton', 'originally from', 'Illawarra')
('Elvis', 'originally from', 'American')
('Pamela Jain', 'originally from', 'Indian')
('Walter Ulfig', 'originally from', 'German')
('Cry', 'originally from', 'Hungarian')
('Ian Barry', 'originally from', 'Australian')
('Sergio Mimica- Gezzan', 'originally from', 'American')
('Alison Skipper', 'originally from', 'American')
('Wollongong', 'originally from', 'Illawarra')
('Tarcisio Fusco', 'originally from', 'Italian')
('Kannada', 'originally from', 'Malayalam')


In [10]:
dataset_2Wiki[1]

{'_id': '3057c6c4086111ebbd5dac1f6bf848b6',
 'type': 'bridge_comparison',
 'question': 'Do both films The Falcon (Film) and Valentin The Good have the directors from the same country?',
 'context': '[["The Falcon Takes Over", ["The Falcon Takes Over( also known as The Falcon Steps Out), is a 1942 black- and- white mystery film directed by Irving Reis.", "The B film was the third, following\\" The Gay Falcon\\" and\\" A Date with the Falcon\\"( 1941), to star George Sanders as the character Gay Lawrence, a gentleman detective known by the sobriquet the Falcon."]], ["The Falcon Strikes Back", ["The Falcon Strikes Back( The Falcon Comes Back) is a 1943 American crime film directed by Edward Dmytryk and stars Tom Conway as the title character, the amateur sleuth, the Falcon.", "Supporting roles are filled by Harriet Hilliard, Jane Randolph, Edgar Kennedy, with Cliff Edwards filling in for Allen Jenkins as the Falcon\'s sidekick,\\" Goldie\\" Locke.", "It is the sixth film in the Falcon ser

In [11]:
print(json.loads(dataset_2Wiki[1]['context']))

[['The Falcon Takes Over', ['The Falcon Takes Over( also known as The Falcon Steps Out), is a 1942 black- and- white mystery film directed by Irving Reis.', 'The B film was the third, following" The Gay Falcon" and" A Date with the Falcon"( 1941), to star George Sanders as the character Gay Lawrence, a gentleman detective known by the sobriquet the Falcon.']], ['The Falcon Strikes Back', ['The Falcon Strikes Back( The Falcon Comes Back) is a 1943 American crime film directed by Edward Dmytryk and stars Tom Conway as the title character, the amateur sleuth, the Falcon.', 'Supporting roles are filled by Harriet Hilliard, Jane Randolph, Edgar Kennedy, with Cliff Edwards filling in for Allen Jenkins as the Falcon\'s sidekick," Goldie" Locke.', 'It is the sixth film in the Falcon series and the second for Conway, reprising the role that his brother, George Sanders had initiated.']], ['Vatroslav Mimica', ['Vatroslav Mimica( born 25 June 1923) is a Croatian film director and screenwriter.', '

# Ver las que faltan de las preguntas por que no las ha detectado

# Entity linker

Pasar a nombre del pais (Frenc - France)

In [1]:
import spacy

In [3]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
doc = nlp("This is a sentence French American.")
# entity_linker = nlp.add_pipe("entity_linker")
entity_linker = nlp.add_pipe("entityLinker", last = True)
# This usually happens under the hood
processed = entity_linker(doc)

In [18]:
doc = nlp("This is a sentence French.")
processed = entity_linker(doc)

In [19]:
for entity in doc._.linkedEntities:
    entity.pretty_print()

<EntityElement: https://www.wikidata.org/wiki/Q41796 sentence                  textual unit consisting of one or more words that are grammatically linked, expressing a complete th>
<EntityElement: https://www.wikidata.org/wiki/Q150 French                    Romance language                                  >


In [33]:
for entity in doc._.linkedEntities:
    # El linker te da la descripción y el ID de Wikidata
    print(f"Texto: {entity.get_span()}")
    print(f"Etiqueta: {entity.get_label()}")
    print(f"ID Wikidata: {entity.get_id()}")
    print(f"ID Wikidata: {entity.get_property('P27')}")
    print("-" * 20)

Texto: sentence
Etiqueta: sentence
ID Wikidata: 41796


AttributeError: 'EntityElement' object has no attribute 'get_property'

In [21]:
for entity in doc._.linkedEntities:
    gentilicio = entity.get_label()  # Nombre en Wikidata (ej: "French")
    origen = entity.get_span()        # Texto en la frase (ej: "French")
    
    # 3. Buscar la entidad madre o superior (Super Entity)
    # Buscamos en su árbol genealógico de Wikidata cuál es el país de origen
    super_entidades = entity.get_super_entities()
    
    pais_encontrado = "No identificado"
    for super_entidad in super_entidades:
        # Imprimimos para entender qué devuelve (opcional)
        # print(f"Hijo de: {super_entidad.get_label()} ({super_entidad.get_description()})")
        
        # Filtro inteligente: buscamos palabras clave en la descripción de Wikidata
        desc = super_entidad.get_description().lower()
        # if "country" in desc or "sovereign state" in desc or "republic" in desc:
        #     pais_encontrado = super_entidad.get_label()
            # break # Nos quedamos con la primera coincidencia válida
        pais_encontrado = super_entidad.get_label()
            
    print(f"Gentilicio en texto: '{origen}' -> País mapeado: {pais_encontrado}")
    print("-" * 40)

Gentilicio en texto: 'sentence' -> País mapeado: semantic unit
----------------------------------------
Gentilicio en texto: 'French' -> País mapeado: Oïl languages
----------------------------------------


In [8]:
doc = nlp("I watched the Pirates of the Caribbean last silvester")

# returns all entities in the whole document
all_linked_entities = doc._.linkedEntities
# iterates over sentences and prints linked entities
for sent in doc.sents:
    sent._.linkedEntities.pretty_print()

<EntityElement: https://www.wikidata.org/wiki/Q194318 Pirates of the Caribbean  Series of fantasy adventure films                 >
<EntityElement: https://www.wikidata.org/wiki/Q664609 Caribbean                 region to the center-east of America composed of many islands / coastal regions surrounding the Cari>
<EntityElement: https://www.wikidata.org/wiki/Q12525597 Silvester                 the day celebrated on 31 December (Roman Catholic Church) or 2 January (Eastern Orthodox Churches)>


In [17]:
texto = "The French chef met the American actor and a Mexican director."
doc = nlp(texto)

# 2. Iterar por cada entidad enlazada detectada en el texto
for entity in doc._.linkedEntities:
    gentilicio = entity.get_label()  # Nombre en Wikidata (ej: "French")
    origen = entity.get_span()        # Texto en la frase (ej: "French")
    
    # 3. Buscar la entidad madre o superior (Super Entity)
    # Buscamos en su árbol genealógico de Wikidata cuál es el país de origen
    super_entidades = entity.get_super_entities()
    
    pais_encontrado = "No identificado"
    for super_entidad in super_entidades:
        # Imprimimos para entender qué devuelve (opcional)
        # print(f"Hijo de: {super_entidad.get_label()} ({super_entidad.get_description()})")
        
        # Filtro inteligente: buscamos palabras clave en la descripción de Wikidata
        desc = super_entidad.get_description().lower()
        if "country" in desc or "sovereign state" in desc or "republic" in desc:
            pais_encontrado = super_entidad.get_label()
            break # Nos quedamos con la primera coincidencia válida
            
    print(f"Gentilicio en texto: '{origen}' -> País mapeado: {pais_encontrado}")
    print("-" * 40)

Gentilicio en texto: 'chef' -> País mapeado: No identificado
----------------------------------------
Gentilicio en texto: 'actor' -> País mapeado: No identificado
----------------------------------------
Gentilicio en texto: 'director' -> País mapeado: No identificado
----------------------------------------


In [ ]:
import spacy

# nlp = spacy.load("en_core_web_sm")
# nlp.add_pipe("entity_linker", last=True)

# Prueba con casos complejos (incluyendo el conflictivo 'American')
texto = "The American programmer met the British engineer and a French chef."
doc = nlp(texto)


In [24]:
doc

This is a sentence French.

In [25]:
processed = entity_linker(doc)

In [26]:
processed

This is a sentence French.

In [30]:

for entity in processed._.linkedEntities:
    gentilicio = entity.get_label()
    print(gentilicio)
    # CONSULTA DIRECTA A WIKIDATA:
    # P27 significa 'country of citizenship' (País de ciudadanía)
    info_ciudadania = entity.get_property("P27")
    
    if info_ciudadania:
        # get_property devuelve un diccionario de Wikidata. 
        # Tomamos el nombre/etiqueta del nodo del país.
        pais_mapeado = info_ciudadania.get("label", "No identificado")
    else:
        pais_mapeado = "No identificado"
        
    print(f"Gentilicio: {gentilicio.ljust(12)} -> País Oficial: {pais_mapeado}")

sentence


AttributeError: 'EntityElement' object has no attribute 'get_property'

## Con pycountry

In [23]:
import pycountry

def encontrar_pais(gentilicio):
    try:
        # Busca el gentilicio en la base de datos de subdivisiones o países
        resultado = pycountry.countries.search_fuzzy(gentilicio)
        if resultado:
            return resultado[0].name  # Devuelve el nombre oficial en inglés
    except LookupError:
        return "No encontrado"

# Pruebas
print(f"French -> {encontrar_pais('French')}")
print(f"American -> {encontrar_pais('American')}")
print(f"Mexican -> {encontrar_pais('Mexican')}")
print(f"Spanish -> {encontrar_pais('Spanish')}")

French -> France
American -> American Samoa
Mexican -> Mexico
Spanish -> Bahamas


## Con country converter

In [1]:
import country_converter as coco

# Inicializas el convertidor
cc = coco.CountryConverter()

# Le pasas una lista o un solo gentilicio
gentilicios = ["French", "American", "Mexican", "Spanish"]
paises = cc.convert(names=gentilicios, to='name_short')

for g, p in zip(gentilicios, paises):
    print(f"{g} -> {p}")

French not found in regex
American not found in regex
Spanish not found in regex


French -> not found
American -> not found
Mexican -> Mexico
Spanish -> not found
